In [2]:
import os
print(os.getcwd())

c:\Users\Lenovo\Desktop\Agai Project\notebooks


In [3]:
os.chdir("C:/Users/Lenovo/Desktop/Agai Project")
print(os.getcwd())

C:\Users\Lenovo\Desktop\Agai Project


In [4]:
import os

audio_folder = "data/raw/audio_samples"
files = os.listdir(audio_folder)
print("Number of items:", len(files))
print(files[:10])

Number of items: 25
['Actor_01', 'Actor_02', 'Actor_03', 'Actor_04', 'Actor_05', 'Actor_06', 'Actor_07', 'Actor_08', 'Actor_09', 'Actor_10']


In [5]:
actor_1_files = os.listdir("data/raw/audio_samples/Actor_01")
print(len(actor_1_files))
print(actor_1_files[:10])

60
['03-01-01-01-01-01-01.wav', '03-01-01-01-01-02-01.wav', '03-01-01-01-02-01-01.wav', '03-01-01-01-02-02-01.wav', '03-01-02-01-01-01-01.wav', '03-01-02-01-01-02-01.wav', '03-01-02-01-02-01-01.wav', '03-01-02-01-02-02-01.wav', '03-01-02-02-01-01-01.wav', '03-01-02-02-01-02-01.wav']


In [6]:
import sys
!{sys.executable} -m pip install librosa soundfile

   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 1.0/1.0 MB 4.6 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   ----------------------------- ---------- 2.1/2.8 MB 10.0 MB/s eta 0:00:01
   ---------------------------------------- 2.8/2.8 MB 9.3 MB/s eta 0:00:00
   ---------------------------------------- 0.0/41.9 MB ? eta -:--:--
   - -------------------------------------- 1.6/41.9 MB 8.8 MB/s eta 0:00:05
   --- ------------------------------------ 3.1/41.9 MB 8.0 MB/s eta 0:00:05
   ---- ----------------------------------- 5.0/41.9 MB 7.9 MB/s eta 0:00:05
   ------ --------------------------------- 6.6/41.9 MB 8.0 MB/s eta 0:00:05
   -------- ------------------------------- 8.4/41.9 MB 8.0 MB/s eta 0:00:05
   --------- ------------------------------ 10.0/41.9 MB 8.0 MB/s eta 0:00:04
   ----------- ---------------------------- 11.8/41.9 MB 8.0 MB/s eta 0:00:04
   ------------- -----


[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import librosa

sample_path = f"data/raw/audio_samples/Actor_01/{actor_1_files[0]}"
audio, sr = librosa.load(sample_path, sr=None)  # sr=None keeps original sample rate

print("Sample rate:", sr)
print("Duration (seconds):", len(audio) / sr)
print("Number of samples:", len(audio))

Sample rate: 48000
Duration (seconds): 3.3032916666666665
Number of samples: 158558


In [8]:
import os
import pandas as pd

emotion_map = {
    '01': 'neutral', '02': 'calm', '03': 'happy', '04': 'sad',
    '05': 'angry', '06': 'fearful', '07': 'disgust', '08': 'surprised'
}

records = []
base_path = "data/raw/audio_samples"

for actor_folder in os.listdir(base_path):
    actor_path = os.path.join(base_path, actor_folder)
    if not os.path.isdir(actor_path):
        continue
    for fname in os.listdir(actor_path):
        if not fname.endswith('.wav'):
            continue
        parts = fname.replace('.wav', '').split('-')
        emotion_code = parts[2]
        records.append({
            'file_path': os.path.join(actor_path, fname),
            'actor': actor_folder,
            'emotion': emotion_map.get(emotion_code, 'unknown'),
            'intensity': 'strong' if parts[3] == '02' else 'normal'
        })

audio_index = pd.DataFrame(records)
print(audio_index.shape)
print(audio_index['emotion'].value_counts())
audio_index.head()

(1440, 4)
emotion
calm         192
happy        192
sad          192
angry        192
disgust      192
fearful      192
surprised    192
neutral       96
Name: count, dtype: int64


,file_path,actor,emotion,intensity
0,data/raw/audio_samples\Actor_01\03-01-01-01-01...,Actor_01,neutral,normal
1,data/raw/audio_samples\Actor_01\03-01-01-01-01...,Actor_01,neutral,normal
2,data/raw/audio_samples\Actor_01\03-01-01-01-02...,Actor_01,neutral,normal
3,data/raw/audio_samples\Actor_01\03-01-01-01-02...,Actor_01,neutral,normal
4,data/raw/audio_samples\Actor_01\03-01-02-01-01...,Actor_01,calm,normal


In [9]:
confidence_map = {
    'calm': 'confident',
    'neutral': 'confident',
    'happy': 'confident',
    'sad': 'nervous',
    'fearful': 'nervous',
    'disgust': 'nervous',
    'angry': 'nervous',
    'surprised': 'neutral'   # ambiguous — could go either way, keep separate
}

audio_index['confidence_label'] = audio_index['emotion'].map(confidence_map)
print(audio_index['confidence_label'].value_counts())

confidence_label
nervous      768
confident    480
neutral      192
Name: count, dtype: int64


In [11]:
import librosa
import numpy as np

TARGET_SR = 16000  # standard for speech models

def extract_audio_features(file_path):
    try:
        # Load and resample to 16kHz mono
        audio, sr = librosa.load(file_path, sr=TARGET_SR, mono=True)
        
        # Trim leading/trailing silence
        audio_trimmed, _ = librosa.effects.trim(audio, top_db=20)
        
        if len(audio_trimmed) == 0:
            return None
        
        # MFCCs (13 coefficients, averaged over time)
        mfccs = librosa.feature.mfcc(y=audio_trimmed, sr=sr, n_mfcc=13)
        mfccs_mean = np.mean(mfccs, axis=1)
        
        # Pitch (fundamental frequency)
        pitches, magnitudes = librosa.piptrack(y=audio_trimmed, sr=sr)
        pitch_values = pitches[magnitudes > np.median(magnitudes)]
        pitch_mean = np.mean(pitch_values) if len(pitch_values) > 0 else 0
        
        # Energy / RMS
        rms = librosa.feature.rms(y=audio_trimmed)
        energy_mean = np.mean(rms)
        
        # Speaking rate proxy (zero-crossing rate)
        zcr = librosa.feature.zero_crossing_rate(audio_trimmed)
        zcr_mean = np.mean(zcr)
        
        # Duration after trimming
        duration = len(audio_trimmed) / sr
        
        return {
            **{f'mfcc_{i}': mfccs_mean[i] for i in range(13)},
            'pitch_mean': pitch_mean,
            'energy_mean': energy_mean,
            'zcr_mean': zcr_mean,
            'duration': duration
        }
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return None

In [12]:
test_features = extract_audio_features(audio_index.iloc[0]['file_path'])
print(test_features)

{'mfcc_0': np.float32(-450.4743), 'mfcc_1': np.float32(114.354576), 'mfcc_2': np.float32(-13.819466), 'mfcc_3': np.float32(31.798847), 'mfcc_4': np.float32(0.47914726), 'mfcc_5': np.float32(-10.455821), 'mfcc_6': np.float32(-13.727076), 'mfcc_7': np.float32(-29.941748), 'mfcc_8': np.float32(-19.479649), 'mfcc_9': np.float32(0.110514835), 'mfcc_10': np.float32(-13.51227), 'mfcc_11': np.float32(3.165472), 'mfcc_12': np.float32(-15.365469), 'pitch_mean': np.float32(1215.9513), 'energy_mean': np.float32(0.0059477696), 'zcr_mean': np.float64(0.09354248046875), 'duration': 1.248}


In [13]:
from tqdm import tqdm

feature_rows = []
for idx, row in tqdm(audio_index.iterrows(), total=len(audio_index)):
    features = extract_audio_features(row['file_path'])
    if features is not None:
        features['file_path'] = row['file_path']
        features['actor'] = row['actor']
        features['emotion'] = row['emotion']
        features['confidence_label'] = row['confidence_label']
        feature_rows.append(features)

audio_features_df = pd.DataFrame(feature_rows)
print(audio_features_df.shape)
audio_features_df.head()

100%|██████████| 1440/1440 [00:56<00:00, 25.66it/s]

(1440, 21)


,mfcc_0,mfcc_1,mfcc_2,mfcc_3,mfcc_4,mfcc_5,mfcc_6,mfcc_7,mfcc_8,mfcc_9,...,mfcc_11,mfcc_12,pitch_mean,energy_mean,zcr_mean,duration,file_path,actor,emotion,confidence_label
0,-450.474304,114.354576,-13.819466,31.798847,0.479147,-10.455821,-13.727076,-29.941748,-19.479649,0.110515,...,3.165472,-15.365469,1215.951294,0.005948,0.093542,1.248,data/raw/audio_samples\Actor_01\03-01-01-01-01...,Actor_01,neutral,confident
1,-455.095642,104.489876,-9.153913,35.398399,-1.780124,-9.343888,-14.313224,-29.996695,-15.997981,0.926548,...,1.051788,-16.181673,1177.948486,0.006189,0.089402,1.312,data/raw/audio_samples\Actor_01\03-01-01-01-01...,Actor_01,neutral,confident
2,-434.738953,107.870682,-5.857293,28.316214,-5.300733,-4.380546,-17.697212,-27.712086,-21.200949,0.239634,...,4.205322,-15.931656,1099.880127,0.007248,0.115942,1.248,data/raw/audio_samples\Actor_01\03-01-01-01-02...,Actor_01,neutral,confident
3,-440.976349,102.615906,-0.556120,29.056000,-0.871811,-1.726907,-20.798180,-26.503531,-18.231178,1.134747,...,-0.783894,-15.475899,1221.080444,0.006734,0.117137,1.216,data/raw/audio_samples\Actor_01\03-01-01-01-02...,Actor_01,neutral,confident
4,-504.640717,108.104530,0.754794,29.923216,2.722944,-5.476415,-14.303703,-19.719933,-20.347515,1.883062,...,3.238217,-13.980220,1243.680420,0.003943,0.109874,1.472,data/raw/audio_samples\Actor_01\03-01-02-01-01...,Actor_01,calm,confident


In [14]:
from sklearn.preprocessing import StandardScaler

feature_cols = [col for col in audio_features_df.columns if col not in 
                ['file_path', 'actor', 'emotion', 'confidence_label']]

scaler = StandardScaler()
audio_features_scaled = audio_features_df.copy()
audio_features_scaled[feature_cols] = scaler.fit_transform(audio_features_df[feature_cols])

print(audio_features_scaled[feature_cols].describe().loc[['mean', 'std']])

            mfcc_0        mfcc_1        mfcc_2        mfcc_3        mfcc_4  \
mean  8.881784e-17  1.134895e-16  1.011537e-16 -4.934325e-17  7.401487e-17   
std   1.000347e+00  1.000347e+00  1.000347e+00  1.000347e+00  1.000347e+00   

            mfcc_5        mfcc_6        mfcc_7        mfcc_8        mfcc_9  \
mean -1.973730e-17 -1.381611e-16  7.894919e-17  8.881784e-17  4.440892e-17   
std   1.000347e+00  1.000347e+00  1.000347e+00  1.000347e+00  1.000347e+00   

       mfcc_10       mfcc_11       mfcc_12    pitch_mean   energy_mean  \
mean  0.000000  3.947460e-17 -3.947460e-17 -2.911251e-16  6.661338e-17   
std   1.000347  1.000347e+00  1.000347e+00  1.000347e+00  1.000347e+00   

          zcr_mean      duration  
mean -2.911251e-16 -5.427757e-17  
std   1.000347e+00  1.000347e+00  


In [15]:
from sklearn.model_selection import train_test_split

audio_train, audio_temp = train_test_split(
    audio_features_scaled, test_size=0.30, stratify=audio_features_scaled['confidence_label'], random_state=42
)
audio_val, audio_test = train_test_split(
    audio_temp, test_size=0.50, stratify=audio_temp['confidence_label'], random_state=42
)

print("Train:", audio_train.shape)
print("Val:", audio_val.shape)
print("Test:", audio_test.shape)

Train: (1008, 21)
Val: (216, 21)
Test: (216, 21)


In [16]:
audio_features_df.to_csv("data/processed/audio/audio_features_raw.csv", index=False)
audio_train.to_csv("data/processed/audio/train.csv", index=False)
audio_val.to_csv("data/processed/audio/val.csv", index=False)
audio_test.to_csv("data/processed/audio/test.csv", index=False)

print("Audio datasets saved to data/processed/audio/")

Audio datasets saved to data/processed/audio/
